# 🏭 AI-Enabled Assets Performance & Predictive Maintenance Platform

This notebook allows you to run the full **Industrial Risk AI** platform directly in Google Colab.


### Step 1: 🚀 Setup Environment
Run this cell to install the platform and dependencies.

In [ ]:
import os
import subprocess
import time
import sys

# 1. Clone Repository
REPO_URL = "https://github.com/lmudu2/industrial-risk-ai.git"
REPO_DIR = "industrial-risk-ai"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

print(f"Cloning {REPO_URL}...")
!git clone {REPO_URL}
os.chdir(f"/content/{REPO_DIR}")

# 2. Install Dependencies
print("Installing dependencies (this may take 2 minutes)...")
!pip install -v -r requirements.txt

import streamlit
print(f"\n✅ Installed Streamlit version: {streamlit.__version__}")
if streamlit.__version__ < "1.34.0":
    print("❌ ERROR: Streamlit version is too low for this project.")
    print("👉 GO TO MENU: 'Runtime' -> 'Restart Session' and run this cell again!")
else:
    print("✅ Version check passed!")

### Step 2: 📊 Generate Industrial Database
Run this cell to generate the synthesized SQLite database.

In [ ]:
if not os.path.exists("backend/eam_database.db"):
    print("Generating real-world industrial data (3-5 minutes)...")
    !python data/generate_data.py
else:
    print("✅ Database already exists.")

### Step 3: ⚡ Start & Monitor Logs
This cell starts the platform and displays **Live Error Logs** below.

In [ ]:
from google.colab import userdata
import subprocess
import time
import os
import threading

# Kill old processes
!pkill streamlit
!pkill uvicorn
!pkill cloudflared

# 0. Load Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded!")
except:
    print("⚠️ GROQ_API_KEY not found in Secrets.")

# 1. Start Services and capture logs
print("Starting Backend and Frontend...")
with open("streamlit.log", "w") as f:
    subprocess.Popen(["streamlit", "run", "frontend/app.py", "--server.port", "8501", "--server.address", "0.0.0.0", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], 
                     stdout=f, stderr=subprocess.STDOUT)

subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])

time.sleep(5)

# 2. Start Cloudflare Tunnel
print("Opening Secure Cloudflare Tunnel...")
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], 
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("\n--- DASHBOARD ACCESS ---")
for line in tunnel.stdout:
    if ".trycloudflare.com" in line:
        import re
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print(f"🚀 SECURE DASHBOARD LINK: {url}")
            print("="*60 + "\n")
            break

print("\n--- LIVE STREAMLIT ERROR LOGS (Check below if 'Oh no' appears) ---")
def tail_logs():
    with open("streamlit.log", "r") as f:
        # Print what's there now
        print(f.read())
        # Then wait for new lines
        while True:
            line = f.readline()
            if line:
                # Hide internal Streamlit URLs to avoid confusion
                # Strictly filter out setup/networking junk
                junk_patterns = ["Local URL:", "Network URL:", "External URL:", "  http://", "view your Streamlit app", "statistics"]
                if not any(pattern in line for pattern in junk_patterns):
                    print(line, end="")
            else:
                time.sleep(1)

thread = threading.Thread(target=tail_logs, daemon=True)
thread.start()

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopping...")